# 12 — Fine-tune ESM2 End-to-End

**Goal**: Instead of just using frozen pooled embeddings as features, we add a classification head to the `facebook/esm2_t6_8M_UR50D` model (for speed) and fine-tune the whole model.

Fine-tuning often pushes AUC by 1–3 points compared to frozen embeddings.

In [ ]:
import pandas as pd
import numpy as np
import torch
from datasets import Dataset
from transformers import AutoTokenizer, EsmForSequenceClassification, Trainer, TrainingArguments
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.model_selection import StratifiedGroupKFold
import warnings
warnings.filterwarnings('ignore')

MODEL_NAME = "facebook/esm2_t6_8M_UR50D"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


## Step 12.1 — Prepare Dataset

In [ ]:
train_df = pd.read_csv('../data/processed/train_clusters.csv')
# Ensure labels are 0 and 1
train_df['Label'] = np.where(train_df['Label'] == 1, 1, 0)

def tokenize_function(examples):
    return tokenizer(examples["Sequence"], padding="max_length", truncation=True, max_length=200)

# Create huggingface dataset
hf_dataset = Dataset.from_pandas(train_df[['Sequence', 'Label']])
tokenized_dataset = hf_dataset.map(tokenize_function, batched=True)
tokenized_dataset = tokenized_dataset.rename_column("Label", "labels")

## Step 12.2 — Define Metrics & Training Loop

In [ ]:
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    # preds are logits. Convert to probabilities for AUC.
    probs = torch.nn.functional.softmax(torch.tensor(predictions), dim=-1)[:, 1].numpy()
    preds_class = np.argmax(predictions, axis=1)
    return {
        'accuracy': accuracy_score(labels, preds_class),
        'auc': roc_auc_score(labels, probs)
    }

# Set up a 1-fold test for fast iteration (can be wrapped in Cluster CV later)
cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
train_idx, val_idx = next(cv.split(train_df, train_df['Label'], groups=train_df['Cluster']))

train_split = tokenized_dataset.select(train_idx)
val_split = tokenized_dataset.select(val_idx)

model = EsmForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

training_args = TrainingArguments(
    output_dir="../models/esm_finetuned",
    learning_rate=1e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=5,
    weight_decay=0.01,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_split,
    eval_dataset=val_split,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

print('Ready to fine-tune ESM2. Call trainer.train() when running on a GPU cluster!')
